In [0]:
# =========================
# CONFIGURATION
# =========================

BASE_PATH = "/Volumes/tmp/tmp/brd_agent"

NOTEBOOKS_PATH = f"{BASE_PATH}/training/notebooks"
BRDS_PATH      = f"{BASE_PATH}/training/brds"
MAPPING_PATH   = f"{BASE_PATH}/training/mappings.csv"

DELTA_NOTEBOOK_CHUNKS = f"{BASE_PATH}/delta/notebook_chunks"
DELTA_BRD_CHUNKS      = f"{BASE_PATH}/delta/brd_chunks"

print("Configuration loaded")


Configuration loaded


In [0]:
%pip install python-docx
import docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 28.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
import uuid
import pandas as pd
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.types import *


In [0]:
# loading mappings.csv

mappings_pd = pd.read_csv(MAPPING_PATH)

required_cols = {"notebook_file", "brd_file", "domain", "schema"}
missing = required_cols - set(mappings_pd.columns)

if missing:
    raise Exception(f"Missing required columns in mappings.csv: {missing}")

if mappings_pd.isnull().any().any():
    raise Exception("mappings.csv contains NULL values. Fix before proceeding.")

print(f"Loaded mappings.csv with {len(mappings_pd)} rows")
display(mappings_pd)


Loaded mappings.csv with 10 rows


notebook_file,brd_file,domain,schema
PDM_Monthly_Production.ipynb,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,BI_Reporting_S4,Production_Operations
Fed_Indian_Venture.ipynb,BRD_BI_Reporting_S4_Land_Fed_Indian_Venture.docx,BI_Reporting,Land
Wellbore_Completion_Downtime.ipynb,BRD_BI_Reporting_S4_Enterprise_Wellbore_Completion_Downtime.docx,BI_Reporting_S4,Enterprise
Plant_Location.ipynb,BRD_BI_Reporting_S4_Supply_Chain_Plant_Location.docx,BI_Reporting-S4,Supply_Chain
Ofm_Daily_Completion_Production.ipynb,BRD_BI_Reporting_S4_Reporting_Ofm_Daily_Completion_Production.docx,BI_Reporting_S4,Reporting
Sap_Bpc_Cost_Center.ipynb,BRD_BI_Reporting_Planning_Budget_SAP_BPC_Cost_Center.docx,BI_Reporting,Planning_Budget
ZFIGL_O14.ipynb,BRD_BI_Reporting_Finance_And_Accounting_ZFIGLO14.docx,BI_Reporting,Finanace_And_Accounting
Work_Order.ipynb,BRD_BI_Reporting_Equipment_Work_Order.docx,BI_Reporting,Equipment
SAPCONTRACTOR.ipynb,BRD_BI_Reporting_EHS_sapcontractor.docx,BI_Reporting,EHS
AP_Invoice.ipynb,BRD_BI_Reporting_Accounts_payable_Ap_Invoice.docx,BI_Reporting,Accounts_Payable


In [0]:
# parse .ipynb notebook

def parse_ipynb(notebook_path):
    with open(notebook_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    parsed_cells = []

    for idx, cell in enumerate(nb.get("cells", [])):
        cell_type = cell.get("cell_type")
        source = "".join(cell.get("source", [])).strip()

        if not source:
            continue

        parsed_cells.append({
            "cell_index": idx,
            "cell_type": cell_type,
            "content": source
        })

    return parsed_cells


In [0]:
# creating notebook chunks -- notebook chunks delta

import json
import uuid
import pandas as pd

# Parse .ipynb notebook content from a string
def parse_ipynb(notebook_content):
    nb = json.loads(notebook_content)
    parsed_cells = []
    for idx, cell in enumerate(nb.get("cells", [])):
        cell_type = cell.get("cell_type")
        source = "".join(cell.get("source", [])).strip()
        if not source:
            continue
        parsed_cells.append({
            "cell_index": idx,
            "cell_type": cell_type,
            "content": source
        })
    return parsed_cells

# Ingest notebook -- notebook chunks delta --
notebook_rows = []

for _, row in mappings_pd.iterrows():
    nb_file = row["notebook_file"]
    domain = row["domain"]
    schema = row["schema"]

    nb_path = f"{NOTEBOOKS_PATH}/{nb_file}"

    # Use dbutils to check file existence
    try:
        dbutils.fs.ls(nb_path)
    except Exception:
        raise Exception(f"Notebook not found: {nb_path}")

    # Read file content using dbutils.fs.head()
    file_content = dbutils.fs.head(nb_path, 10485760)  # Read up to 10MB

    cells = parse_ipynb(file_content)

    for cell in cells:
        chunk_type = "markdown" if cell["cell_type"] == "markdown" else "code"
        notebook_rows.append({
            "chunk_id": str(uuid.uuid4()),
            "notebook_name": nb_file,
            "schema": schema,
            "domain": domain,
            "chunk_type": chunk_type,
            "cell_index": cell["cell_index"],
            "content": cell["content"]
        })

notebook_df = spark.createDataFrame(
    pd.DataFrame(notebook_rows)
)

notebook_df.write.format("delta").mode("overwrite").save(DELTA_NOTEBOOK_CHUNKS)

display(notebook_df)

chunk_id,notebook_name,schema,domain,chunk_type,cell_index,content
6a8665fe-90c8-4b4e-b9bb-c9fbbec8a1e7,PDM_Monthly_Production.ipynb,Production_Operations,BI_Reporting_S4,markdown,0,"- BI Reporting : PDM_MONTHLY_PRODUCTION - Description : This notebook constructs Monthly Production related data by using various entity/parameter tables and final table is materialized under Interface_Work schema.The Notebook will provide monthly production volumetric data corresponding to each entity using multiple table for various product like oil and gas with various 2nd disposition type like(SALES TO TRANSMISSION COMPANY, SALES TO SPOT MARKET, SALES TO TRUCK,SALES TO OIL PIPELINE). - Domain : Production Operations - Precondition : N/A"
e2140ea2-51a3-41be-a5a9-74ad52ab7ee6,PDM_Monthly_Production.ipynb,Production_Operations,BI_Reporting_S4,code,1,from pyspark.sql.types import * from pyspark.sql.functions import *
83051e19-fba0-436e-a64b-f90d52bb0f8b,PDM_Monthly_Production.ipynb,Production_Operations,BI_Reporting_S4,markdown,2,###Inheriting Connection and common functions notebooks
c92015e4-b0e9-4f7e-9715-fe78be03a0c6,PDM_Monthly_Production.ipynb,Production_Operations,BI_Reporting_S4,code,3,%run /Apps/BIReporting_S4/Config/SnowflakeConnection
ea8232dd-396b-4599-89d7-9ec8de2a9688,PDM_Monthly_Production.ipynb,Production_Operations,BI_Reporting_S4,code,4,%run /Apps/BIReporting_S4/Utils/CommonFunctions
3c6f59a7-9f5a-4499-a700-39b11f3e9cea,PDM_Monthly_Production.ipynb,Production_Operations,BI_Reporting_S4,markdown,5,### Setting Source and Target Variables
6b01a561-a8ae-422b-96b3-9cc4fb63191f,PDM_Monthly_Production.ipynb,Production_Operations,BI_Reporting_S4,code,6,"#source Variables var_src_pmp_table_unit = ""PVCALCUNITSUS.PVUNIT"" var_src_pmp_table_comp = ""PVCALCUNITSUS.PVUNITCOMP"" var_src_pmp_table_allocm = ""PVCALCUNITSUS.PVUNITALLOCMONTH"" var_src_pmp_table_comppram = ""PVCALCUNITSUS.PVUNITCOMPPARAM"" var_src_pmp_table_unitnode = ""PVCALCUNITSUS.PVUNITNODE"" var_src_pmp_table_unitdispm = ""PVCALCUNITSUS.PVUNITDISPMONTH"" #list for various product and corresponding type value var_pmp_distyp_list = ['Gas','SALES TO TRANSMISSION COMPANY', 'Gas','SALES TO SPOT MARKET', 'Oil','SALES TO TRUCK', 'Oil','SALES TO OIL PIPELINE'] #target variables var_tgt_pmp_schema = ""INTERFACE_WORK"" var_tgt_pmp_table = ""PDM_MONTHLY_PRODUCTION"" var_tgt_pmp_table_id = ""PRODUCTION_OPERATIONS.PDM_MONTHLY_PRODUCTION"""
626f3d56-44f0-479e-b311-44fecd3df9a3,PDM_Monthly_Production.ipynb,Production_Operations,BI_Reporting_S4,markdown,7,## Functions definition
0af17b87-235e-412c-bebb-d63718a82d49,PDM_Monthly_Production.ipynb,Production_Operations,BI_Reporting_S4,code,8,"#function is creatd to handle the repeated logic we have to calculate the Oil/Sales volumes for different conditions def typedisp_2 (prd_name,typdisp_val, i ): #conditional logic which helps in considering which column to parse to the returning dataframe if prd_name == 'Oil': colval = ""VOLHCLIQ"" elif prd_name == 'Gas': colval = ""VOLGas"" #join between table Unit and Unit Node with various various disposition type df_pmp_unit_unode_jn = df_pmp_pv_unit.join(df_pmp_pv_unitnode,[df_pmp_pv_unit.IDREC == df_pmp_pv_unitnode.IDRECPARENT,\ df_pmp_pv_unitnode.DISPOSITIONPOINT == 1,\ df_pmp_pv_unitnode.DISPPRODUCTNAME == prd_name,\ upper(df_pmp_pv_unitnode.TYPDISP1) == 'SALE',\ upper(df_pmp_pv_unitnode.TYPDISP2) == typdisp_val],\ how= 'inner')\ .select(df_pmp_pv_unitnode.IDREC)\ .drop(df_pmp_pv_unitnode.IDRECPARENT) #join of above dataframe with table Unit Disposition Month to have supportive factors for further calculation df_pmp_unode_udm_jn = df_pmp_unit_unode_jn.join(df_pmp_pv_udisp_mon,\ [df_pmp_unit_unode_jn.IDREC == df_pmp_pv_udisp_mon.IDRECDISPUNITNODE],\ how= 'inner')\ .select(df_pmp_pv_udisp_mon.IDRECCOMP,""DTTMEND"",""VOLGas"",""VOLHCLIQ"") #join of above dataframe with table Unit Comp to have PDEN ID with respect to product and dispostion type 2 df_pmp_undm_com_jn = df_pmp_unode_udm_jn.join(df_pmp_pv_uni

In [0]:
# validate notebook ingestion

df_nb = spark.read.format("delta").load(DELTA_NOTEBOOK_CHUNKS)

print("Notebook chunk count:", df_nb.count())
display(df_nb.orderBy("notebook_name", "cell_index"))


Notebook chunk count: 334


chunk_id,notebook_name,schema,domain,chunk_type,cell_index,content
3a7fa88c-ecb0-4fea-a631-f0d48b39adc5,AP_Invoice.ipynb,Accounts_Payable,BI_Reporting,markdown,0,- BI Reporting : AP_Invoice - Description : This table provides Ap_Invoice related complete information. This component is materialized under AccountsPayable schema. - Domain : AccountsPayable - Precondition : All the source Tables should have latest data - Postcondition : N/A
82992741-cd94-4248-81a2-96acaa377d22,AP_Invoice.ipynb,Accounts_Payable,BI_Reporting,code,1,from pyspark.sql.functions import * from pyspark.sql.types import *
45396482-d048-4411-adaf-ff67cf747558,AP_Invoice.ipynb,Accounts_Payable,BI_Reporting,markdown,2,####Inheriting Connection and common functions notebooks
96aea0b9-b1c1-46da-92f3-0f603d871812,AP_Invoice.ipynb,Accounts_Payable,BI_Reporting,code,3,%run /Apps/BIReporting/Config/SnowflakeConnection
b9e584f6-418c-4019-bb2b-05bf5b5ea2d5,AP_Invoice.ipynb,Accounts_Payable,BI_Reporting,markdown,4,#### Setting Source and Target Variables
89bb701c-51ea-4c37-9c49-ea5e7244dc6a,AP_Invoice.ipynb,Accounts_Payable,BI_Reporting,code,5,"#source variables var_src_api_lastpay = BIREPORTING_DB + "".ACCOUNTS_PAYABLE.LAST_PAYMENT"" var_src_api_image = BIREPORTING_DB + "".ACCOUNTS_PAYABLE.VIM_NONVIM_IMAGE"" var_src_api_bbh = BIREPORTING_DB + "".ACCOUNTS_PAYABLE.BSAK_BSIK_HEADER"" var_src_api_rollup = BIREPORTING_DB + "".ACCOUNTS_PAYABLE.INVOICE_CLEARING_ROLLUP"" var_src_api_acct = SAP_DB +"".ECC.ZAP_NON_VIM_ACCT"" #Target Variable var_tgt_api_schema = ""ACCOUNTS_PAYABLE"" var_tgt_api_id = ""ACCOUNTS_PAYABLE.AP_INVOICE"" var_tgt_api_table = ""AP_INVOICE"""
03a3e90b-c999-4730-be6e-43e0bec06d6e,AP_Invoice.ipynb,Accounts_Payable,BI_Reporting,markdown,6,####Reading source tables
1af5abc3-186c-4464-b626-9e1e29fa2f5f,AP_Invoice.ipynb,Accounts_Payable,BI_Reporting,code,7,"#Reading LAST_PAYMENT table from snowflake df_api_last_payment = spark.read.format(""snowflake"")\ .options(**sfOptionsBISOX_Read)\ .option(""dbtable"",var_src_api_lastpay)\ .load()\ .select(col(""BUKRS""), col(""AUGBL"").alias(""_PAYMENT_CLEARING_DOCUMENT_NUMBER""), col(""PAYMENT_FISCAL_YEAR"").alias(""_PAYMENT_FISCAL_YEAR""), col(""BELNR""), col(""AMT_LAST_PMT_WITHHOLD_TAX_DOC""), col(""AMT_LAST_PMT_DISC_ELIG_CORP""), col(""AMT_LAST_PMT_DISC_ELIG_DOC""), col(""AMT_LAST_PMT_DISC_ELIG_LOCAL""), col(""AMT_LAST_PMT_DISC_LOST_CORP""), col(""AMT_LAST_PMT_DISC_LOST_DOC""), col(""AMT_LAST_PMT_DISC_LOST_LOCAL""), col(""AMT_LAST_PMT_DISC_TAKEN_CORP""), col(""AMT_LAST_PMT_DISC_TAKEN_DOC""), col(""AMT_LAST_PMT_DISC_TAKEN_LOCAL""), col(""AMT_LAST_PMT_PAID_NET_CORP""), col(""AMT_LAST_PMT_PAID_NET_DOC""), col(""AMT_LAST_PMT_PAID_NET_LOCAL""), col(""ZLSCH""), col(""ZTERM""), col(""LAST_PAYMENT_PAYMENT_KEY""), col(""LAST_PAYMENT_PMT_METHOD_SUP_KEY""), col(""LAST_PAYMENT_PMT_TERM_KEY""), col(""DISCOUNT_LOST_FLAG"").alias(""_DISCOUNT_LOST_FLAG""), col(""CALENDAR_PAYMENT_KEY"").alias(""_CALENDAR_PAYMENT_KEY""), col(""AMT_TOTAL_PAID_NET_CORPORATE""), col(""AMT_TOTAL_PAID_NET_DOCUMENT""), col(""AMT_TOTAL_PAID_NET_LOCAL""))"
196275b2-cdae-4e15-8d02-60971eb3ffd8,AP_Invoice.ipynb,Accounts_Payable,BI_Reporting,code,8,"#Reading VIM_NONVIM_IMAGE table from snowflake df_api_vim_nonvim_image = spark.read.format(""snowflake"")\ .options(**sfOptionsBISOX_Read)\ .option(""dbtable"",var_src_api_image)\ .load()"
306039fc-7052-46d3-a82d-25621060807b,AP_Invoice.ipynb,Accounts_Payable,BI_Reporting,code,9,"#Reading INVOICE_CLEARING_ROLLUP table from snowflake df_api_clearing_rollup = spark.read.format(""snowflake"")\ .options(**sfOptionsBISOX_Read)\ .option(""dbtable"",var_src_api_rollup)\ .load()\ .select(col(""BUKRS""), col(""BELNR""), col(""GJAHR""), col(""CLEARING_DOCUMENT_NUMBER"").alias(""CL_CLEARING_DOCUMENT_NUMBER""), col(""CALENDAR_CLEARING"").alias(""CL_CALENDAR_CLEARING""))"


In [0]:
from docx import Document
import tempfile
import os

def parse_brd_docx_from_volume_spark(brds_dir, target_filename):
    """
    Reads a DOCX from a Unity Catalog Volume using Spark binaryFile reader.
    Works with READ-only permissions.
    """

    df = (
        spark.read
        .format("binaryFile")
        .option("pathGlobFilter", "*.docx")
        .load(brds_dir)
        .filter(F.col("path").endswith(target_filename))
    )

    rows = df.select("content").collect()

    if len(rows) == 0:
        raise Exception(f"DOCX not found via Spark in directory: {target_filename}")

    binary_content = rows[0]["content"]

    with tempfile.NamedTemporaryFile(delete=False, suffix=".docx") as tmp:
        tmp.write(binary_content)
        tmp_path = tmp.name

    try:
        doc = Document(tmp_path)
        sections = []

        current_section = "UNCLASSIFIED"
        buffer = []

        for para in doc.paragraphs:
            text = para.text.strip()
            if not text:
                continue

            # Heading heuristic (same as before)
            if text.isupper() or text[:1].isdigit():
                if buffer:
                    sections.append({
                        "section": current_section,
                        "content": "\n".join(buffer)
                    })
                    buffer = []

                current_section = text
            else:
                buffer.append(text)

        if buffer:
            sections.append({
                "section": current_section,
                "content": "\n".join(buffer)
            })

        return sections

    finally:
        os.remove(tmp_path)


In [0]:
brd_rows = []   # brd chunking with sparx as dbutils didnt work with docx in uc  -- brd chunk delta 

BRDS_DIR = BRDS_PATH  # directory, NOT file

for _, row in mappings_pd.iterrows():
    brd_file = row["brd_file"]
    domain   = row["domain"]

    sections = parse_brd_docx_from_volume_spark(
        brds_dir=BRDS_DIR,
        target_filename=brd_file
    )

    for sec in sections:
        brd_rows.append({
            "chunk_id": str(uuid.uuid4()),
            "brd_name": brd_file,
            "section": sec["section"],
            "domain": domain,
            "content": sec["content"]
        })

brd_df = spark.createDataFrame(pd.DataFrame(brd_rows))

brd_df.write.format("delta").mode("overwrite").save(DELTA_BRD_CHUNKS)

display(brd_df)
print("✅ BRD ingestion completed successfully using Spark binaryFile (directory mode)")


chunk_id,brd_name,section,domain,content
62e793bf-b06f-49c4-b46c-c17795884e86,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,UNCLASSIFIED,BI_Reporting_S4,Production Operations – PDM_Monthly_Production Functional and Technical Design Contents Contents 2
49cd266e-3f47-41de-a31c-d91dbe570285,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,DOCUMENT CONTROL,BI_Reporting_S4,Document Audience Document History Approval History
071bcf57-2cc4-42c9-9926-b37f48a0f8ac,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,OVERVIEW AND SCOPE,BI_Reporting_S4,Functional Description
ea5c4243-f6df-456e-8b7a-03ea2d44aa40,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,SUMMARY OF WHAT WE ARE TRYING TO ACCOMPLISH –,BI_Reporting_S4,"We are bringing granular level pdm monthly production volumetric data corresponding to each entity using multiple table for various product like oil and gas with various 2nd disposition type like(SALES TO TRANSMISSION COMPANY, SALES TO SPOT MARKET, SALES TO TRUCK,SALES TO OIL PIPELINE). Legacy Information Tidal DAS_IS_LANDING_HANA_pdm_monthly_injection"
92c2561c-9657-4a66-a006-3b08d884d74d,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,N/A,BI_Reporting_S4,"Informatica Workflow: wkf_HANA_tow_facility_5 Mappings: m_HANA_PV_pdm_monthly_prod_ins_BIOPS Source qualifier SQL query: select Unit_Comp.IDRECPARENT AS PDEN_ID, 'Well' as PDEN_TYPE, Null as ZONE_NAME, 'ProdView' AS DATA_SOURCE, TO_DATE(Month_Alloc.DtTmENd) AS PROD_DT, Null as PRODUCTION_HDR_ID, Month_Alloc.DurOp*60*24 AS PT_MI_ON, Null as TOT_NO_WELL, Null As VO_OIL_ALLOWABLE , Month_Alloc.VolProdAllocOil AS VO_OIL_PROD, Month_Alloc.VolProdAllocCond AS VO_COND_PROD, NULL as VO_GAS_ALLOWABLE, Month_Alloc.VolProdAllocGas AS VO_GAS_PROD, Null as VO_GCH_PROD , Null as VO_GWG_PROD , Month_Alloc.VolProdAllocWater AS VO_WAT_PROD, Month_Alloc.VolStartInvHCLiq AS VO_OIL_BEG_INV, Month_Alloc.VolEndInvHCLiq AS VO_OIL_END_INV, Null as VO_OIL_FLD_GLOS, Null as VO_OIL_FLD_OTHER, Null as VO_OIL_INTER_OIL , Null as VO_OIL_INTRA_OTHER, Null as VO_OIL_RCV_TREAT , Null as VO_OIL_SALE_BARGE , Null as VO_OIL_SALE_INTER , Null as VO_OIL_SALE_INTRA , CASE --WHEN TO_DATE(Month_Alloc.DtTmENd)< TO_DATE('07/01/2021') THEN Month_Alloc.VolDispSaleHCLiq"
64e6a96a-f56d-4075-a32c-295dd58ffe58,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,WHEN ((OIL_PIPELINE.VO_OIL_SALE_PIPE IS NULL OR OIL_PIPELINE.VO_OIL_SALE_PIPE =0),BI_Reporting_S4,AND (OIL_SALE_TRUCK.VO_OIL_SALE_TR IS NULL OR OIL_SALE_TRUCK.VO_OIL_SALE_TR =0)) THEN Month_Alloc.VolDispSaleHCLiq
855abaa4-55c2-4263-815e-fb1b1d544146,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,"ELSE OIL_PIPELINE.VO_OIL_SALE_PIPE END AS VO_OIL_SALE_PIPE,",BI_Reporting_S4,"Null as VO_OIL_SALE_RR ,"
bef50b5e-bc3e-4e3f-8023-f8d1a2ebe081,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,"OIL_SALE_TRUCK.VO_OIL_SALE_TR AS VO_OIL_SALE_TR,",BI_Reporting_S4,"Null as VO_OIL_TREAT , Null as VO_WAT_RCV_TREAT, Null as VO_WAT_SUPPLY , Null as VO_WAT_TREAT , Null as VO_GAS_SALE_INTER , Null as VO_GAS_SALE_INTRA , CASE --WHEN TO_DATE(Month_Alloc.DtTmENd)< TO_DATE('07/01/2021') THEN Month_Alloc.VolDispSaleGas"
6d3b9d22-9f29-4ab7-92d0-dda4015439f9,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,WHEN ((GAS_SALE_PIPE.VO_GAS_SALE_PIPE IS NULL OR GAS_SALE_PIPE.VO_GAS_SALE_PIPE =0 ),BI_Reporting_S4,AND (GAS_SALE_SPOT.VO_GAS_SALE_SPOT IS NULL OR GAS_SALE_SPOT.VO_GAS_SALE_SPOT =0 )) THEN Month_Alloc.VolDispSaleGas
97c763ea-c216-4351-ad0a-7cf64ca10325,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,"ELSE GAS_SALE_PIPE.VO_GAS_SALE_PIPE END AS VO_GAS_SALE_PIPE,",BI_Reporting_S4,"Null as VO_GAS_SALE_PLANT , Null as VO_GAS_SALE_RESID ,"


✅ BRD ingestion completed successfully using Spark binaryFile (directory mode)


In [0]:
# validate brd ingestion

df_brd = spark.read.format("delta").load(DELTA_BRD_CHUNKS)

print("BRD chunk count:", df_brd.count())
display(df_brd)


BRD chunk count: 165


chunk_id,brd_name,section,domain,content
62e793bf-b06f-49c4-b46c-c17795884e86,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,UNCLASSIFIED,BI_Reporting_S4,Production Operations – PDM_Monthly_Production Functional and Technical Design Contents Contents 2
49cd266e-3f47-41de-a31c-d91dbe570285,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,DOCUMENT CONTROL,BI_Reporting_S4,Document Audience Document History Approval History
071bcf57-2cc4-42c9-9926-b37f48a0f8ac,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,OVERVIEW AND SCOPE,BI_Reporting_S4,Functional Description
ea5c4243-f6df-456e-8b7a-03ea2d44aa40,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,SUMMARY OF WHAT WE ARE TRYING TO ACCOMPLISH –,BI_Reporting_S4,"We are bringing granular level pdm monthly production volumetric data corresponding to each entity using multiple table for various product like oil and gas with various 2nd disposition type like(SALES TO TRANSMISSION COMPANY, SALES TO SPOT MARKET, SALES TO TRUCK,SALES TO OIL PIPELINE). Legacy Information Tidal DAS_IS_LANDING_HANA_pdm_monthly_injection"
92c2561c-9657-4a66-a006-3b08d884d74d,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,N/A,BI_Reporting_S4,"Informatica Workflow: wkf_HANA_tow_facility_5 Mappings: m_HANA_PV_pdm_monthly_prod_ins_BIOPS Source qualifier SQL query: select Unit_Comp.IDRECPARENT AS PDEN_ID, 'Well' as PDEN_TYPE, Null as ZONE_NAME, 'ProdView' AS DATA_SOURCE, TO_DATE(Month_Alloc.DtTmENd) AS PROD_DT, Null as PRODUCTION_HDR_ID, Month_Alloc.DurOp*60*24 AS PT_MI_ON, Null as TOT_NO_WELL, Null As VO_OIL_ALLOWABLE , Month_Alloc.VolProdAllocOil AS VO_OIL_PROD, Month_Alloc.VolProdAllocCond AS VO_COND_PROD, NULL as VO_GAS_ALLOWABLE, Month_Alloc.VolProdAllocGas AS VO_GAS_PROD, Null as VO_GCH_PROD , Null as VO_GWG_PROD , Month_Alloc.VolProdAllocWater AS VO_WAT_PROD, Month_Alloc.VolStartInvHCLiq AS VO_OIL_BEG_INV, Month_Alloc.VolEndInvHCLiq AS VO_OIL_END_INV, Null as VO_OIL_FLD_GLOS, Null as VO_OIL_FLD_OTHER, Null as VO_OIL_INTER_OIL , Null as VO_OIL_INTRA_OTHER, Null as VO_OIL_RCV_TREAT , Null as VO_OIL_SALE_BARGE , Null as VO_OIL_SALE_INTER , Null as VO_OIL_SALE_INTRA , CASE --WHEN TO_DATE(Month_Alloc.DtTmENd)< TO_DATE('07/01/2021') THEN Month_Alloc.VolDispSaleHCLiq"
64e6a96a-f56d-4075-a32c-295dd58ffe58,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,WHEN ((OIL_PIPELINE.VO_OIL_SALE_PIPE IS NULL OR OIL_PIPELINE.VO_OIL_SALE_PIPE =0),BI_Reporting_S4,AND (OIL_SALE_TRUCK.VO_OIL_SALE_TR IS NULL OR OIL_SALE_TRUCK.VO_OIL_SALE_TR =0)) THEN Month_Alloc.VolDispSaleHCLiq
855abaa4-55c2-4263-815e-fb1b1d544146,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,"ELSE OIL_PIPELINE.VO_OIL_SALE_PIPE END AS VO_OIL_SALE_PIPE,",BI_Reporting_S4,"Null as VO_OIL_SALE_RR ,"
bef50b5e-bc3e-4e3f-8023-f8d1a2ebe081,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,"OIL_SALE_TRUCK.VO_OIL_SALE_TR AS VO_OIL_SALE_TR,",BI_Reporting_S4,"Null as VO_OIL_TREAT , Null as VO_WAT_RCV_TREAT, Null as VO_WAT_SUPPLY , Null as VO_WAT_TREAT , Null as VO_GAS_SALE_INTER , Null as VO_GAS_SALE_INTRA , CASE --WHEN TO_DATE(Month_Alloc.DtTmENd)< TO_DATE('07/01/2021') THEN Month_Alloc.VolDispSaleGas"
6d3b9d22-9f29-4ab7-92d0-dda4015439f9,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,WHEN ((GAS_SALE_PIPE.VO_GAS_SALE_PIPE IS NULL OR GAS_SALE_PIPE.VO_GAS_SALE_PIPE =0 ),BI_Reporting_S4,AND (GAS_SALE_SPOT.VO_GAS_SALE_SPOT IS NULL OR GAS_SALE_SPOT.VO_GAS_SALE_SPOT =0 )) THEN Month_Alloc.VolDispSaleGas
97c763ea-c216-4351-ad0a-7cf64ca10325,BRD_BI_Reporting_S4_Production_Operations_PDM_Monthly_Production.docx,"ELSE GAS_SALE_PIPE.VO_GAS_SALE_PIPE END AS VO_GAS_SALE_PIPE,",BI_Reporting_S4,"Null as VO_GAS_SALE_PLANT , Null as VO_GAS_SALE_RESID ,"


In [0]:
# final validation

print("==== FINAL VALIDATION ====")
print("Notebook files:", df_nb.select("notebook_name").distinct().count())
print("BRD files:", df_brd.select("brd_name").distinct().count())

assert df_nb.count() > 0, "No notebook chunks created"
assert df_brd.count() > 0, "No BRD chunks created"

print("✅ INGESTION SUCCESSFUL")


==== FINAL VALIDATION ====
Notebook files: 10
BRD files: 10
✅ INGESTION SUCCESSFUL
